# Lesson 4: Bring an existing Brain-Score model

**You will learn:** how models registered for classic Brain-Score become subjects, and how to translate one into a `BrainScoreModel`.
**Requires:** CPU only, under a minute.

Classic vision models are `BrainModel` objects with methods such as `look_at` and `start_recording`; language models are `ArtificialSubject` objects with `digest_text`. You do not rewrite them: `brainscore.load_model(...)` wraps them in an **adapter** that presents them as a subject, and classic benchmarks still run on any subject.

**The check:** does the classic registration and its translation record exactly the same activity from the same network?

In [1]:
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)   # no progress bars
warnings.filterwarnings('ignore')                            # keep the lesson output short

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images (CIFAR-10 test set, 12 per category), committed with the lessons.
ASSETS = Path('../assets/lessons')
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

In [3]:
import torch
import torchvision
from PIL import Image

# SqueezeNet 1.1 trained on ImageNet (committed, 5 MB). It never saw these images.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()

TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files to a (batch, 3, 224, 224) array, as the network was trained."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

## A classic vision registration

In [4]:
from brainscore_vision.model_helpers.activations.pytorch import PytorchWrapper
from brainscore_vision.model_helpers.brain_transformation import ModelCommitment

legacy_model = ModelCommitment(
    identifier='squeezenet-classic',
    activations_model=PytorchWrapper(identifier='squeezenet-classic', model=net, preprocessing=preprocess),
    layers=['features.12'],
    region_layer_map={'IT': 'features.12'},
)

## The adapter makes it a subject

In [5]:
from brainscore_vision.compat.unified_adapter import VisionModelAdapter
from brainscore_core.streaming import BufferedSession

classic = VisionModelAdapter(legacy_model)
print('outputs:', classic.out_channels)
session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:IT'], time_bins=[(70, 170)])
classic.participate(session)
legacy_activity = session.collect()
legacy_activity.shape

outputs: {'behavior', 'neural:IT'}


(120, 86528)

## Translating the registration

| Classic | `BrainScoreModel` |
| --- | --- |
| `PytorchWrapper(..., preprocessing=...)` | `VisionWrapper(..., preprocessing=...)` in `extractor={'vision': ...}` |
| `region_layer_map={'IT': ...}` | `region_layer_map={'IT': ...}` |
| `look_at(stimuli)` | `participate(session)` (or `process(stimuli)` interactively) |

In [6]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper

translated = BrainScoreModel(
    identifier='squeezenet-new', model=net,
    extractor={'vision': VisionWrapper(identifier='squeezenet-new', model=net, preprocessing=preprocess)},
    region_layer_map={'IT': 'features.12'})
session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:IT'])
translated.participate(session)
new_activity = session.collect()
legacy_values = np.asarray(legacy_activity.squeeze().values)
print('same activity:', np.allclose(np.sort(new_activity.values, axis=None), np.sort(legacy_values, axis=None)))

same activity: True


The two record the same numbers from the same network: translating changes the interface, not the measurement.

## Loading by name

Registered models need no wrapping: `brainscore.load_model('alexnet')` returns a subject, classic or new, and `brainscore.score(model, benchmark)` runs any benchmark on it.

## Summary

- Adapters make classic models subjects; classic benchmarks run on any subject.
- A translation is one to one, and it records the same activity.

**Next:** [Lesson 5](05_layers_and_regions.ipynb) compares layers against a null network.